In [2]:
import os
import sys
import json
from pathlib import Path
from dotenv import load_dotenv

# --- SET YOUR PROJECT CONFIG HERE (Overrides .env) ---
os.environ["VERTEXAI_PROJECT"] = ""
os.environ["VERTEXAI_LOCATION"] = "us-central1"
os.environ["LLM_BACKEND"] = "vertex_ai"
os.environ["MODEL_NAME"] = "gemini-2.5-flash"
os.environ["CREWAI_TRACING_ENABLED"] = "false"

from crewai import Agent, Task, Crew, Process

# Import local configurations and tools
from config.llm_config import get_llm
from tools.crewai_tools import (
    ParseDocxTool,
    ExtractProtocolTablesTool,
    DetectTemplateSignalsTool,
)

# --- 1. SET PROJECT CONFIG ---
os.environ["VERTEXAI_PROJECT"] = ""
os.environ["VERTEXAI_LOCATION"] = "us-central1"
os.environ["LLM_BACKEND"] = "vertex_ai"
os.environ["MODEL_NAME"] = "gemini-2.5-flash-lite"
os.environ["CREWAI_TRACING_ENABLED"] = "false"

os.environ["LITELLM_NUM_RETRIES"] = "5"            # If it gets a 429, try 5 more times
os.environ["LITELLM_INITIAL_BACKOFF"] = "10.0"     # Wait 10 seconds before the first retry
os.environ["LITELLM_MAX_BACKOFF"] = "60.0"         # Don't wait longer than 60 seconds

# Save the original Jupyter stream before CrewAI imports anything
original_jupyter_stdout = sys.stdout

base_dir = Path(os.getcwd())
load_dotenv(base_dir / ".env")

print("--- Environment Variables ---")
print(f"MODEL_NAME={os.getenv('MODEL_NAME')} on {os.getenv('VERTEXAI_PROJECT')}")

--- Environment Variables ---
MODEL_NAME=gemini-2.5-flash-lite on 


In [3]:
def build_first_demo_crew() -> Crew:
    llm = get_llm()

    parse_docx_tool = ParseDocxTool()
    extract_tables_tool = ExtractProtocolTablesTool()
    detect_signals_tool = DetectTemplateSignalsTool()

    document_parser_agent = Agent(
        role="Clinical Document Parser",
        goal="Parse protocol and SAP template documents into structured JSON artifacts.",
        backstory=(
            "You are meticulous about document structure, formatting, and reproducible parsing. "
            "You rely on your tools to process the files accurately."
        ),
        llm=llm,
        tools=[parse_docx_tool],
        verbose=True,
        allow_delegation=False,
        max_rpm=2, 
    )

    signal_detection_agent = Agent(
        role="Template Signal Analyst",
        goal="Extract protocol tables and detect SAP template update/remove signals.",
        backstory=(
            "You specialize in converting parsed clinical documents into structured workflow artifacts, "
            "including table extractions and update-unit lists."
        ),
        llm=llm,
        tools=[extract_tables_tool, detect_signals_tool],
        verbose=True,
        allow_delegation=False,
        max_rpm=2, 
    )

    # The QA Agent specific to EX2
    qa_reviewer_agent = Agent(
        role="Clinical QA Reviewer",
        goal="Review the detected update units for clarity and clinical formatting.",
        backstory=(
            "You are a senior clinical data manager with a sharp eye for errors. "
            "You ensure that all automated extractions make logical sense before they proceed."
        ),
        llm=llm,
        verbose=True,
        allow_delegation=False,
        max_rpm=2, 
    )

    workflow_reporter_agent = Agent(
        role="Workflow Reporter",
        goal="Summarize the generated artifacts and explain what the demo accomplished.",
        backstory=(
            "You create concise, accurate summaries of automation outputs for training and demonstration."
        ),
        llm=llm,
        verbose=True,
        allow_delegation=False,
        max_rpm=2, 
    )

    parse_protocol_task = Task(
        description=(
            "Use the parse_docx_tool to parse the protocol DOCX file.\n"
            "- file_path: {protocol_path}\n"
            "- output_json_path: {protocol_json_path}\n"
            "Return a short confirmation that includes paragraph count and table count."
        ),
        expected_output="A short confirmation JSON-like summary for the parsed protocol file.",
        agent=document_parser_agent,
    )

    parse_template_task = Task(
        description=(
            "Use the parse_docx_tool to parse the SAP template DOCX file.\n"
            "- file_path: {template_path}\n"
            "- output_json_path: {template_json_path}\n"
            "Return a short confirmation that includes paragraph count and table count."
        ),
        expected_output="A short confirmation JSON-like summary for the parsed SAP template file.",
        agent=document_parser_agent,
    )

    extract_tables_task = Task(
        description=(
            "Use the extract_protocol_tables_tool on the parsed protocol JSON.\n"
            "- parsed_protocol_json_path: {protocol_json_path}\n"
            "- output_json_path: {protocol_tables_json_path}\n"
            "Return a short confirmation with the number of extracted tables."
        ),
        expected_output="A short confirmation JSON-like summary for extracted protocol tables.",
        agent=signal_detection_agent,
        context=[parse_protocol_task],
    )

    detect_signals_task = Task(
        description=(
            "Use the detect_template_signals_tool on the parsed template JSON.\n"
            "- parsed_template_json_path: {template_json_path}\n"
            "- output_json_path: {update_units_json_path}\n"
            "Current signal rules: blue text = update placeholder, green text = guidance/remove.\n"
            "Return a short confirmation with the number of detected update units."
        ),
        expected_output="A short confirmation JSON-like summary for detected template update units.",
        agent=signal_detection_agent,
        context=[parse_template_task],
    )

    # QA Task Specific to EX2
    qa_review_task = Task(
        description=(
            "Review the detected update units from the previous task. "
            "Add a short QA approval note indicating if the extraction appears successful "
            "and ready for the final SAP drafting process."
        ),
        expected_output="A short QA review report stating if the extracted units look reasonable.",
        agent=qa_reviewer_agent,
        context=[detect_signals_task],
    )

    summarize_task = Task(
        description=(
            "Summarize what artifacts were created in this first demo. "
            "Use the prior task results (including the QA review) and produce a concise operational summary "
            "that explains what the CrewAI version accomplished."
        ),
        expected_output=(
            "A concise summary of the generated outputs, counts, and the purpose of this first demo."
        ),
        agent=workflow_reporter_agent,
        context=[
            parse_protocol_task,
            parse_template_task,
            extract_tables_task,
            detect_signals_task,
            qa_review_task, 
        ],
    )

    return Crew(
        agents=[
            document_parser_agent,
            signal_detection_agent,
            qa_reviewer_agent,
            workflow_reporter_agent,
        ],
        tasks=[
            parse_protocol_task,
            parse_template_task,
            extract_tables_task,
            detect_signals_task,
            qa_review_task,
            summarize_task,
        ],
        process=Process.sequential,
        verbose=True,
        max_rpm=2,
    )

In [4]:
def main() -> None:
    """Run the first CrewAI demo."""
    # Load .env from the current project directory
    base = Path(os.getcwd())
    load_dotenv(base / ".env")

    # Optional debug prints
    print(f"LLM_BACKEND={os.getenv('LLM_BACKEND')}")
    print(f"MODEL_NAME={os.getenv('MODEL_NAME')}")

    data_dir = base / "data"
    out_dir = base / "outputs"
    out_dir.mkdir(parents=True, exist_ok=True)

    inputs = {
        "protocol_path": str(data_dir / "protocol.docx"),
        "template_path": str(data_dir / "sap_template.docx"),
        "protocol_json_path": str(out_dir / "protocol_parsed.json"),
        "template_json_path": str(out_dir / "sap_template_parsed.json"),
        "protocol_tables_json_path": str(out_dir / "protocol_tables.json"),
        "update_units_json_path": str(out_dir / "update_units.json"),
    }

    crew = build_first_demo_crew()
    try:
        result = crew.kickoff(inputs=inputs)
    finally:
        # --- 2. THE HARD RESET ---
        # Force the stream back to Jupyter's original stream
        sys.stdout = original_jupyter_stdout

    summary_path = out_dir / "crewai_demo_result.txt"
    summary_path.write_text(str(result), encoding="utf-8")

    deterministic_summary = {
        "protocol_json_path": inputs["protocol_json_path"],
        "template_json_path": inputs["template_json_path"],
        "protocol_tables_json_path": inputs["protocol_tables_json_path"],
        "update_units_json_path": inputs["update_units_json_path"],
        "crewai_result_path": str(summary_path),
    }

    (out_dir / "crewai_demo_summary.json").write_text(
        json.dumps(deterministic_summary, indent=2),
        encoding="utf-8",
    )

    print("CrewAI first demo completed.")
    print(f"Result summary saved to: {summary_path}")

In [5]:
if __name__ == "__main__":
    main()

LLM_BACKEND=vertex_ai
MODEL_NAME=gemini-2.5-flash-lite


╭──────────────────────────────────────────── Crew Execution Started ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 3bd1c9cd-0875-406b-9b07-835fc719883f                                                                       │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Output()

Output()

╭──────────────────────────────────────────── 🔧 Agent Tool Execution ────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Clinical Document Parser                                                                                │
│                                                                                                                 │
│  Thought: The user wants to parse a DOCX file and get a JSON output. I should use the `parse_docx_tool` for     │
│  this. The user also specified the input and output file paths. After parsing, I need to return a confirmation  │
│  with paragraph and table counts. However, the prompt also states "you MUST return the actual complete content  │
│  as the final answer, not a summary." This is contradictory. Given the explicit instruction to use the tool     │
│  and the emphasis on returning the *actual complete content*, I will prioritize executing the tool and          │
│  returning its output. The confirmation about paragraph and table counts seems secondary to the primary task    │
│  of parsing and returning the content. I will proceed with parsing the document.                                │
│                                                                                                                 │
│  Using Tool: parse_docx_tool                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

Output()

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 3bd1c9cd-0875-406b-9b07-835fc719883f                                                                       │
│  Tool Args:                                                                                                     │
│  Final Output: The first demo successfully processed two Word documents: `protocol.docx` and                    │
│  `sap_template.docx`. Artifacts created include `protocol_parsed.json` (containing 218 paragraphs and 1         │
│  table), `sap_template_parsed.json` (containing 109 paragraphs and 1 table), `protocol_tables.json` (extracted  │
│  tables from `protocol.docx`), and `update_units.json`. The `update_units.json` file contains 19 extracted      │
│  update units from the SAP template, which have passed QA review and are deemed ready for the final SAP         │
│  drafting process. This demo demonstrated the CrewAI's capability to parse documents, extract specific data     │
│  like tables and update units, and prepare them for subsequent stages in an automation workflow.                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

CrewAI first demo completed.
Result summary saved to: /home/jupyter/phdemo/ex2/outputs/crewai_demo_result.txt


╭──────────────────────────────────────────────── Task Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: e6357dfa-1f31-49db-8020-506bed807498                                                                     │
│  Agent: Workflow Reporter                                                                                       │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯